In [6]:
import utils
import numpy as np
import pandas as pd
import os
import importlib
import glob
import re
import matplotlib.pyplot as plt

# Exclusion criteria
- Check only the last two blocks
- cutoff for congruent = 60%
- cutoff for incongruent = 100% - congruent + 10%

In [7]:
# Preparation
pilot_analysis = False
unique_subject_ids = [3,4,6,7,8,9]#[802,803,804,805,806,807,808,809,810,811,813,814,815]
unique_sessions = [1,2,3,4]
plot_range = [0, 6]

if pilot_analysis:
    raw_output_path = '/Volumes/project/3025011.02/pilot_data/phd_1_task/experiment_output/behavioural_files_pilot_2'
else:
    raw_output_path = '/Volumes/project/3025011.02/raw/'
recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(raw_output_path, unique_subject_ids, unique_sessions)

sub-006 did not complete session-02
sub-006 did not complete session-03
sub-006 did not complete session-04
sub-007 did not complete session-04
sub-008 did not complete session-03
sub-008 did not complete session-04
sub-009 did not complete session-02
sub-009 did not complete session-03
sub-009 did not complete session-04


# Adjust simulations
- Check MI Skin and Brain to see if it's below 1.9
- Check CEM43 skin and brain to see if it's below 0.25

In [8]:
simulation_directory = f'/Volumes/project/3025011.02/TUS_simulations/planning/'

stimulation_output_full_filenames_1 = os.path.join(simulation_directory, '**', '*layered_output_table_target_1*.csv')
stimulation_output_full_filenames_2 = os.path.join(simulation_directory, '**', '*layered_output_table_target_2*.csv')
stimulation_output_full_filenames_3 = os.path.join(simulation_directory, '**', '*layered_output_table_target_3*.csv')

stimulation_output_full_filenames = glob.glob(stimulation_output_full_filenames_1, recursive=True) + glob.glob(stimulation_output_full_filenames_2, recursive=True)
print(stimulation_output_full_filenames)

['/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-004/sub-004_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-008/sub-008_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-003/sub-003_layered_output_table_target_1_6_dc10_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-003/sub-003_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-010/sub-010_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-005/sub-005_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/target_1_6/sub-006/sub-006_layered_output_table_target_1_6_dc20_strength_1.csv', '/Volumes/project/3025011.02/TUS_simulations/planning/

In [9]:
# List to hold individual dataframes
list_of_dataframes = []

# Process each CSV file
for stimulation_output_full_filename in stimulation_output_full_filenames:
    # Extract the filename (without the path)
    stimulation_output_filename = os.path.basename(stimulation_output_full_filename)
    # Remove the '.csv' extension
    stimulation_output_filename_without_extension = os.path.splitext(stimulation_output_filename)[0]

    # Split the filename by underscores
    stimulation_output_filename_parts = stimulation_output_filename_without_extension.split('_')

    # Extract the desired parts from the filename:
    subject_id = stimulation_output_filename_parts[0]                     # e.g. 'sub-010'
    target = '_'.join(stimulation_output_filename_parts[4:7])               # e.g. 'target_2_2'
    duty_cycle = stimulation_output_filename_parts[7]                       # e.g. 'dc1'
    intensity = '_'.join(stimulation_output_filename_parts[8:10])           # e.g. 'strength_1'

    # Read the CSV file (assuming it contains one row)
    try:
        df = pd.read_csv(stimulation_output_full_filename)
    except Exception as e:
        print(f"Error reading {stimulation_output_full_filename}: {e}")
        continue  # Skip this file if there's an error

    # Add the extracted metadata as new columns
    df['target'] = target
    df['intensity'] = intensity
    df['duty_cycle'] = duty_cycle

    # Append the dataframe to our list
    list_of_dataframes.append(df)

# Combine all dataframes into one
if list_of_dataframes:
    stimulation_output = pd.concat(list_of_dataframes, ignore_index=True)
    print(stimulation_output)
else:
    raise Exception("No CSV files were found or processed.")

     subject_id  max_Isppa  max_Isppa_after_exit_plane  real_focal_distance  \
0             4   16.46417                    16.46417            64.509689   
1             8   11.44545                    11.44545            55.056789   
2             3   15.27858                    15.27858            64.048810   
3             3   15.27858                    15.27858            64.048810   
4            10   13.73819                    13.73819            67.503704   
..          ...        ...                         ...                  ...   
135           3   15.05523                    15.05523            43.014532   
136           3   19.40464                    19.40464            43.014532   
137           3   32.52235                    22.31392            42.514703   
138           7   22.52806                    18.99613            43.011626   
139           5   17.76952                    17.76952            42.538218   

     max_Isppa_skin  max_Isppa_skull  max_Isppa_bra

In [10]:
# Reorder columns
column_indices = list(range(stimulation_output.shape[1]))
reordered_column_indices = [column_indices[0]] + column_indices[-3:] + column_indices[1:-3]
stimulation_output = stimulation_output.iloc[:, reordered_column_indices]
stimulation_output.sort_values(by=['subject_id', 'target', 'intensity', 'duty_cycle'], inplace=True)
stimulation_output.reset_index(drop=True, inplace=True)
stimulation_output['CEM43_brain_skin'] = stimulation_output[['CEM43_brain', 'CEM43_skin']].max(axis=1)
# Replace values to improve readability
stimulation_output['duty_cycle'] = stimulation_output['duty_cycle'].map({'dc10': '10%', 'dc15': '15%', 'dc20': '20%'})
stimulation_output['intensity'] = stimulation_output['intensity'].map({'strength_1': '35', 'strength_2': '63', 'strength_3': '81', 'strength_4': '110'})
# Now drop some columns
stimulation_output.drop(
    ['real_focal_distance','Ix_brain', 'Iy_brain','Iz_brain',
     'CEM43_brain', 'CEM43_skull', 'CEM43_skin',
     'focus_pos_final_1', 'focus_pos_final_2', 'focus_pos_final_3',
     'trans_pos_final_1', 'trans_pos_final_2', 'trans_pos_final_3'],
    axis=1,
    inplace=True
)

print(stimulation_output)

     subject_id      target intensity duty_cycle  max_Isppa  \
0             3  target_1_1        35        10%   22.93461   
1             3  target_1_1        35        20%   22.93461   
2             3  target_1_2        35        10%   15.27858   
3             3  target_1_2        35        20%   15.27858   
4             3  target_1_3        35        10%   22.93461   
..          ...         ...       ...        ...        ...   
135          10  target_2_2        81        20%   19.70230   
136          10  target_2_3        81        20%   23.07279   
137          10  target_2_4        81        20%   27.21426   
138          10  target_2_5        81        20%   38.53838   
139          10  target_2_6        81        20%   31.35181   

     max_Isppa_after_exit_plane  max_Isppa_skin  max_Isppa_skull  \
0                      22.93461        22.93461         6.330170   
1                      22.93461        22.93461         6.330170   
2                      15.27858        